# My glacier — analysis notebook (mini-project)

This notebook is your **lab book** for the mini-project (guidelines: Lecture 2 for Part A and B1–B2, Lecture 3 for B3–B4 and the report). It lives in **your own copy** of the template, `~/work/my-glacier/` (copied once from `~/work/glamod/session2/my-glacier/`). It only **reads and plots** the results of your IGM runs: re-run it as often as you like.

**How we work** (as in Session 2):

| What | Where |
|---|---|
| **Set** the parameters | YAML files in `experiment/` (text editor) |
| **Run** IGM | a **Terminal**, from this folder: `cd ~/work/my-glacier` |
| **Look** at the results | this notebook, reading `outputs/<run>/` |

Steps:
0. Set your glacier in `experiment/common/my_glacier.yaml` (`RGI_ID`)
1. **Present**: download the data, check the initial state
2. **Past**: Little Ice Age → today (calibration)
3. **Future**: scenarios 2020–2100 (`run_future.sh`)
4. **Explorations** (choose at least one): B1–B2 in Session 2, B3–B4 in Session 3
5. Figures for the report → `figures/`

⚠️ Copy your notebook, YAML files and figures to your SWITCHdrive folder regularly.

In [ ]:
import sys, os, re, yaml
sys.path.insert(0, os.path.expanduser("~/work/glamod/utils"))   # course plotting helpers (utils/helper.py)
sys.path.insert(0, os.path.abspath("../../utils"))               # (same, when run inside the course folder)
from helper import *

with open("experiment/common/my_glacier.yaml") as f:
    RGI_ID = yaml.safe_load(f)["inputs"]["oggm_shop"]["RGI_ID"]
INPUT = f"data/input_{RGI_ID}.nc"
print("My glacier:", RGI_ID)
os.makedirs("figures", exist_ok=True)                  # figures for the report go here


## Run log

Keep track of every run here (double-click to edit):

| Run folder | Experiment file | What I changed | Why | Result |
|---|---|---|---|---|
| `outputs/check` | `params_future` | 1-year test run | download data | |
| | | | | |

## 1. Present: download the data and check the initial state

▶ **In the terminal**: a 1-year run downloads the data (≈ 1 min the first time) and gives the modelled velocity field of today's glacier:

```bash
igm_run +experiment=params_future processes.time.end=2021 hydra.run.dir=outputs/check
```

In [ ]:
plot_input_data(INPUT)
plot_climate(RGI_ID)

In [ ]:
# Modelled (2020) vs observed surface speeds -> adjust tau_ref / arrhenius in common/my_glacier.yaml if needed
compare_modeled_observed_velocity(INPUT, "outputs/check/output.nc", year=2020)

**Notes** (initial state, observations, present-day climate offset printed above, ...):

- ...

## 2. Past: from the Little Ice Age to today

Start in 1700 from today's geometry, advance to the LIA maximum (~1850), retreat with the observed climate 1901–2019. A 320-year run takes a few minutes.

▶ **In the terminal**: edit `experiment/params_past.yaml` (or a copy, e.g. `params_past_v2.yaml`), then run it into a **new folder** each time:

```bash
igm_run +experiment=params_past hydra.run.dir=outputs/past_v1
```

Then set `run` below to the folder of the run you want to look at, and add it to the comparison.

In [ ]:
run = "outputs/past_v1"
show_dashboard(run, n=4)
analyze_time_series(os.path.join(run, "output_ts.nc"))

In [ ]:
# Ice thickness through time, and along the main flowline
animate_glacier_evolution(os.path.join(run, "output.nc"), variable="thk")
plot_flowline_evolution(os.path.join(run, "output.nc"), rgi_id=RGI_ID)

In [ ]:
# Compare several past runs with today's observed area / volume (from the input data)
ds = xr.open_dataset(INPUT); dx = float(ds.x[1] - ds.x[0])
obs = {"area": ([2020], [float((ds.thk > 1).sum()) * dx**2 / 1e6]),
       "vol":  ([2020], [float(ds.thk.sum()) * dx**2 / 1e9])}
past_runs = ["outputs/past_v1"]              # <- add your other past runs here
compare_runs(past_runs, [os.path.basename(r) for r in past_runs], observations=obs)

✅ Is the LIA maximum reached around 1850? Does the front reach the LIA moraines (compare with the Siegfried map on map.geo.admin.ch)? Is today's extent right? If not, change `delta_temp` (before 1901) or `melt_enhancer`, save as a new file / run folder (`past_v2`, ...), and compare.

**Calibrated values**: `delta_temp` (pre-1901) = ..., `melt_enhancer` = ...

## 3. Future: scenarios 2020–2100

`delta_temp` is relative to **1960–1990** (see Lecture 2, Part A, Step 3):
- 2020 row: present-day offset (printed by `plot_climate` above)
- 2100 row: 1981–2010 offset + CH2018 warming of the scenario

▶ **In the terminal**: open `run_future.sh`, set the 3 numbers at the top (your calibrated `melt_enhancer` and the two offsets printed by `plot_climate`), save, then run the 4 scenarios (committed, RCP2.6, RCP4.5, RCP8.5) one after the other (a few minutes):

```bash
bash run_future.sh
```

Each scenario goes to `outputs/future_<name>/`. The values used are recorded in `outputs/future_<name>/.hydra/overrides.yaml`.

In [ ]:
scenarios = ["committed", "rcp26", "rcp45", "rcp85"]
compare_runs([f"outputs/future_{n}" for n in scenarios], scenarios)

In [ ]:
# Maps for one scenario
animate_glacier_evolution("outputs/future_rcp45/output.nc", variable="thk")

## 4. Explorations (Part B: choose at least one)

Session 2: B1 (SMB model), B2 (ice-flow parameters). Session 3: B3 (response time), B4 (moraines).

### B1 — Two types of SMB model

▶ **In the terminal**: adapt the ELA history in `experiment/params_past_simple.yaml` (today's ELA from the snowline / GLAMOS), then

```bash
igm_run +experiment=params_past_simple hydra.run.dir=outputs/past_simple_v1
```

In [ ]:
plot_smb_simple(ela=3000, gradabl=0.009, gradacc=0.005, accmax=2.0)   # <- your ELA and gradients
compare_runs(["outputs/past_v1", "outputs/past_simple_v1"], ["climate-driven SMB", "ELA-based SMB"], observations=obs)

### B2 — Ice-flow parameters (arrhenius × tau_ref)

▶ **In the terminal** (12 runs of 1 year, ≈ 3 min): `bash run_sensitivity.sh`

In [ ]:
import glob
sens_runs = sorted(glob.glob("outputs/sens_*"))
compare_velocity_misfit(INPUT, sens_runs, year=2020)

### B3 — Response time (your own IGM module, oscillating ELA)

▶ **In the terminal** (400 model years, a few minutes); other periods with `processes.smb_oscillating.period=...`:

```bash
igm_run +experiment=params_response_time hydra.run.dir=outputs/response_200
igm_run +experiment=params_response_time processes.smb_oscillating.period=100 hydra.run.dir=outputs/response_100
```

In [ ]:
plot_response_time("outputs/response_200")
# plot_response_time("outputs/response_100")

### B4 — Moraines with particles

▶ **In the terminal**: put your calibrated `delta_temp` and `melt_enhancer` in `experiment/params_past_moraines.yaml`, then

```bash
igm_run +experiment=params_past_moraines hydra.run.dir=outputs/past_moraines
```

In [ ]:
run_p = "outputs/past_moraines"
for year in [1850, 1950, 2020]:
    plot_particles(run_p, year=year)                # all particles, coloured by depth
plot_particles(run_p, year=2020, rh_min=0.95)       # debris at the surface only: where moraines form

## 5. Figures for the report

The helper functions in `utils/helper.py` are meant for quick looks. For the report, make your own figures: copy the code of a helper into a cell below and adapt it (years shown, titles, units, colors, fonts), then save it into `figures/` **before** `plt.show()`:

```python
plt.savefig("figures/fig_volume.png", dpi=200, bbox_inches="tight")
```

In [ ]:
# your figures here